# InterScale Pipeline: Local Model

Local Model = Graph Convolutional Neural Network

Data: CosmX Pancreas

In [26]:
import scanpy as sc

import InterScale as interscale
from InterScale.config import load_config
from graph_transformer_long_range_niches.pp import split_adata
from InterScale.tl import prepare_geome_dataset, check_and_update_cfg
from InterScale.geome_dataloader import GraphAnnDataModule

# from graph_transformer_long_range_niches.config import load_config
# from graph_transformer_long_range_niches.pp import sliding_window

In [27]:
CFG_PATH_GRAPH = "/home/icb/francesca.drummer/1-Projects/GT-long-range-niches/src/config_files/CosmX_Pancreas/pancreas_graph_sample_gnn.yaml"
CFG_PATH_REG = ""

## 0. Load and prepare data

We load a subset of the CosmX pancreas data containing T1D (type 1 diabetes) and ND (no diabetes) samples and the config file with the model specifications. 

In [28]:
cfg = load_config(CFG_PATH_GRAPH)
cfg

CfgNode({'wandb': CfgNode({'use': False, 'project_name': 'GTLongRange_CosmXPancreas'}), 'model': CfgNode({'n_embed': 32, 'local_component': CfgNode({'name': 'GCN', 'load': None, 'parameters': CfgNode({'embed_dim': 32, 'hidden_dim': 64, 'num_layers': 2, 'dropout_local': 0.0})}), 'global_component': CfgNode({'name': None, 'load': None}), 'save': '/lustre/groups/ml01/projects/2024_spatial_long_range_GT_francesca.drummer/results/cosmx_pancreas/', 'loss': 'GaussianNLL', 'decoder': CfgNode({'type': 'linear', 'hidden_dims': [256, 128], 'dropout_decoder': 0.1})}), 'optim': CfgNode({'lr': 0.001, 'wd': 0.0, 'lr_warmup': 20, 'loss': 'CrossEntropy', 'seed': 44, 'cross_corr': 'gene', 'n_epochs': 4000}), 'dataset': CfgNode({'h5ad_data': '/lustre/groups/ml01/projects/2024_spatial_long_range_GT_francesca.drummer/data/cosmx_pancreas.h5ad', 'name': 'pancreas', 'description': '', 'prediction_task': 'classification', 'prediction_obs': 'condition', 'prediction_level': 'graph', 'layer_key': 'log1p_norm', 's

In [29]:
adata = sc.read_h5ad(cfg.dataset.h5ad_data)
adata

AnnData object with n_obs × n_vars = 386727 × 979
    obs: 'fov', 'Area', 'AspectRatio', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.GCG', 'Max.GCG', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', 'cell_ID', 'condition', 'slide', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_NegPrb', 'log1p_total_counts_NegPrb', 'pct_counts_NegPrb', 'n_genes', 'cell_type_coarse', 'CellTypes_max', 'slide_fov', 'slide_condition', 'split', 'split_0', 'split_1', 'split_2', 'globalX', 'globalY', 'sliding_window_square', 'sliding_window_h', 'sliding_window_v'
    obsm: 'X_pca', 'X_umap', 'spatial', 'spatial_fov'
    layers: 'counts'

In [30]:
import pandas as pd
# Creating a DataFrame from 'split', 'fov', and 'condition'
df = adata.obs[['split', 'condition']]
value_counts = pd.DataFrame(df.values, columns=df.columns).value_counts()
print(value_counts)

split  condition
test   T1D          100230
val    T1D           77807
train  ND            61460
test   ND            51365
val    ND            48614
train  T1D           47251
Name: count, dtype: int64


## 1. Data setup

We need to specify:

- `prediction_task`: Prediction task can either be classification or regression
- `prediction_level`: Which level the predictions should be performed on: either (1) tissue label, e.i. condition (`graph`), (2) node label (`node`) for cell type or niche prediciton, or (3) GEX prediction.

Additionally, we define dataset specific keys: 
- `prediction_obs`: Label in `adata.obs` to be predicted. Only required for classification tasks.
- `layer_key`: Defines which GEX matrix to retrieve from `adata.layer`
- `sample_key`: `adata.obs` used to split the samples into PyG Data objects
- `group_label`: Optional: only if we have a `adata.obs` group that we want to stratify during sampling

In [31]:
PREDICTION_TASK = 'classification'
PREDICTION_LEVEL = 'graph'

prediction_obs = 'condition'
layer_key = 'log1p_norm'
sample_key = 'sliding_window_square'
group_label = 'condition'

In [32]:
cfg = check_and_update_cfg(cfg, prediction_task = PREDICTION_TASK, prediction_level = PREDICTION_LEVEL, layer_key = layer_key, sample_key = sample_key, prediction_obs = prediction_obs)

Update group label (from 'condition' to 'None')


In [33]:
interscale.model.LocalModel._setup_anndata(adata = adata, prediction_task = PREDICTION_TASK, layer_key = layer_key, sample_key = sample_key, prediction_obs = prediction_obs, group_key = group_label)

ValueError: log1p_norm is not a valid key in adata.layers.

In [ ]:
adata

## 2. Model setup

Either set up the model from scratch or load trained model from file (Section 4).

In [ ]:
model = interscale.model.LocalModel(
    adata,
    cfg = cfg
)

In [ ]:
model._model_summary_string

## 3. Training

In [ ]:
#split_adata(adata, split_obs='sample', val_size=cfg.dataset.val_size, test_size=cfg.dataset.test_size, seed = cfg.optim.seed, stratify_groups = cfg.dataset.group_label)
pyg_data_list, _ = prepare_geome_dataset(adata, cfg)
dm = GraphAnnDataModule(datas=pyg_data_list, 
                           num_workers=1, 
                           batch_size=int(cfg.dataset.batch_size), 
                           pct_mask_nodes=cfg.dataset.pct_mask_nodes,
                           learning_type="node")

In [ ]:
# for now training only works with datamodule. TODO: Add DataSplitter for datamodule independent training.
model.train(max_epochs = 100, 
           datamodule = dm,
           early_stopping = True)

In [ ]:
model.history_.plot_history(subset_term = 'loss')
model.history_.plot_history(subset_term = 'r2')
model.history_.plot_history(subset_term = 'pearson')

## 4. Model saving and loading

In [ ]:
model.save('/ictstr01/home/icb/francesca.drummer/1-Projects/GT-long-range-niches/docs/notebooks/model/')

In [ ]:
model = interscale.model.LocalModel.load('/ictstr01/home/icb/francesca.drummer/1-Projects/GT-long-range-niches/docs/notebooks/model/', adata, cfg)

In [ ]:
model

## 5. Evaluation

Evaluation can either be run on the entire AnnData that the model was set up with or subsets of AnnData defined by sample_id from `adata.obs[sample_key]`.

For the LocalModel we get:

- `adata.obsm.local_emb`: Local embedding values for each cell [N,E]
- `adata.obsm.decoder_weight`: Decoder weight values for the cells [N,F]

In [ ]:
sub_adata = adata[adata.obs['condition'] == 'SHH']

In [ ]:
result = model.get_model_output(sub_adata)

In [ ]:
result

### Tissue level analysis

Can we detect a pattern across cells in which are more and less relevant to predict the condition? 

No cell types so need to categorize them differently. 

1. SHH expressed or not (Distance to SHH source)
2. What analysis could we perform without knowing that the SHH is the source for variation? Can we somehow trace the changes back to genes? 

In [ ]:
# plot celltype x condition 
import pandas as pd

def summarise_decoder_weight_by_group_and_condition(
    adata, 
    group_obs: str,
    condition_obs: str = '_scvi_prediction_obs', 
    decoder_weight_key: str = 'decoder_weight', 
):
    """
    Summarizes average attention weights for each (group_obs x condition_obs) combination.
    
    Parameters
    ----------
    adata : AnnData
        AnnData object containing attention weights in adata.obsm[decoder_weight_key].
    decoder_weight_key : str
        Key for the attention weight matrix in adata.obsm.
    condition_obs : str
        Column name in adata.obs for condition (e.g., 'disease_status').
    group_obs : str
        Column name in adata.obs for group (e.g., 'cell_type').
        
    Returns
    -------
    pd.DataFrame
        Multi-index DataFrame with average attention weights for each group x condition.
        Rows: (group_obs, condition_obs)
        Columns: attention per target cell type (from decoder_weight_key).
    """
    # Extract relevant data
    weight = adata.obsm[decoder_weight_key]
    group = adata.obs[group_obs]
    condition = adata.obs[condition_obs]
    
    # Create DataFrame for easy grouping
    df = pd.DataFrame(
        weight, 
        columns=[f"{decoder_weight_key}_{i}" for i in range(weight.shape[1])]
    )
    df[group_obs] = group.values
    df[condition_obs] = condition.values
    
    # Group by group_obs and condition_obs, then average
    grouped = df.groupby([group_obs, condition_obs]).mean()
    
    return grouped

In [ ]:
summarise_decoder_weight_by_group_and_condition(
    result, 
    condition_obs = 'shh_cluster', 
    group_obs = 'condition'
)

In [ ]:
data_list